# Session 6 — DEIM: guided continuation

Download the notebook from the toolbar and use [the course Python environment](https://feelpp.github.io/course-rom/course-rom/setup.html). This is a **guided exercise**: the source data are supplied, but the algorithm is yours to implement. Functions intentionally raise `NotImplementedError` until completed; the starter runs because these exercise functions are not called automatically. Instructor solutions stay outside Antora.


Work one question at a time, show the requested check, then continue after discussion. Keep numerical calculations and plotting in separate cells. The reference graphs in the lecture slides illustrate what to investigate; no reference interpolation basis or completed algorithm is supplied in this notebook.


This extends [the EIM practical](https://feelpp.github.io/course-rom/course-rom/labs/session05-eim.html) on the **same source and snapshots**. Reuse your source data and interpolation solver in your working notebook. The small setup below is repeated only so this downloadable extension also starts independently.
Read [the DEIM steps](https://feelpp.github.io/course-rom/rom/hyper-reduction/deim.html#deim-algorithm).


The Python variable `n` represents the source-grid dimension $N_q$ in the formulas. See [the common notation](https://feelpp.github.io/course-rom/rom/notation.html).
## D1. Use a POD basis of the source (5 minutes)

The POD basis approximates the **source**, not the temperature. The SVD is supplied because today&#8217;s new task is selecting interpolation points for fixed modes. Keep the source fields uncentered. The default rank 8 supports comparison with EIM; later reuse the RIC selection that you implemented for the thermal problem.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
n = 120
x = np.linspace(0., 1., n)
def source(points, center, width):
    return .1+np.exp(-((np.asarray(points)-center)/width)**2)
params = [(c, w) for c in np.linspace(.2, .8, 9) for w in (.08, .14, .2)]
tests = [(.275, .11), (.425, .17), (.675, .13)]
S = np.column_stack([source(x, *p) for p in params])
U, singular_values, Vt = np.linalg.svd(S, full_matrices=False)
m = 8
Um = U[:, :m]
relative_singular_values = singular_values/singular_values[0]
print(f'DEIM POD starter: {n} grid points, {m} source modes; point selection remains to implement.')


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.semilogy(np.arange(1, len(singular_values)+1), relative_singular_values, 'o-')
ax.axvline(m, color='gray', linestyle=':', label='Default comparison rank')
ax.set(xlabel='Mode index', ylabel='Singular value / first singular value', title='POD of the source snapshots')
ax.grid(alpha=.25); ax.legend()
fig.tight_layout(); plt.show()


Check orthonormality of the retained modes. Explain what a source mode represents and why its dimension need not equal the temperature reduced dimension.
## D2. Select points one mode at a time (10–15 minutes)

Select the first coordinate from the first mode. For every next mode, interpolate that mode with the preceding modes and coordinates, then locate the largest remaining component. Work one iteration at a time before writing the loop.


In [ ]:
def deim(basis):
    indices = []
    # TODO: choose the first point from the first mode.
    # TODO: for each next mode, form the small interpolation system.
    # TODO: subtract its interpolant and choose the largest residual magnitude.
    # TODO: reject numerical dependence; preserve previous indices.
    raise NotImplementedError('D2: implement DEIM point selection')

# After implementing D2:
# points = deim(Um)
# T = Um[points, :]
# TODO: verify distinct points and nonsingularity of T.


**Check before continuing:** the new residual vanishes at previous points. Unlike EIM, the matrix of the **original POD modes** is generally not unit triangular. Do not impose that EIM property on this matrix.
## D3. Reuse the online interpolation solver (10 minutes)

Reuse E2/E5 from the EIM practical with the POD basis in place of the EIM basis. If working in this standalone notebook, complete the small interface below. Only sampled source values enter the online solve.


In [ ]:
def deim_online(basis, indices, parameter):
    # TODO: reuse your sampled interpolation solve from EIM.
    raise NotImplementedError('D3: online interpolation on the POD source basis')

# Numerics after point selection works:
# TODO: store POD projection errors and DEIM interpolation errors on the same tests.
# TODO: verify equality with source values at the selected coordinates.


In [ ]:
# Visualization only:
# TODO: plot the stored projection and interpolation errors against rank.
# TODO: compare EIM and DEIM on identical inputs, ranks and a common error norm.


## Checkpoint; optional amplification study (5 minutes)

Explain why POD projection and DEIM interpolation differ. Check that every retained source mode is reproduced by interpolation. Report the selected points and a held-out error plot. If time permits, compute the amplification factor $1/\sigma_{\min}(T)$ and test the Euclidean bound from the notes; its proof is not required today. A source RIC is not a certificate for a new parameter.
The [online PDE experiment](https://feelpp.github.io/course-rom/course-rom/labs/session07-hyper-reduction.html) is a later continuation. The algorithm here already supplies the two ingredients it needs: a term basis and selected sampling coordinates.
